# Module 18: Data Engineering Basics — Solutions

## Complete Solutions for All Exercises

## Part 1: ETL vs ELT — Solutions

In [ ]:
import random
import json
from collections import Counter

# Exercise 1.1: ETL Pipeline
def extract():
    departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance']
    names = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hank']
    data = []
    for i in range(100):
        data.append({
            'name': random.choice(names) + '_' + str(i),
            'age': random.randint(16, 70),
            'salary': random.randint(30000, 150000),
            'department': random.choice(departments)
        })
    return data

def transform(data):
    # Filter adults
    adults = [d for d in data if d['age'] >= 18]
    for person in adults:
        person['tax'] = round(person['salary'] * 0.2, 2)
        if person['salary'] < 50000:
            person['salary_category'] = 'low'
        elif person['salary'] <= 100000:
            person['salary_category'] = 'medium'
        else:
            person['salary_category'] = 'high'
    return adults

def load(data):
    salaries = [d['salary'] for d in data]
    dept_dist = Counter(d['department'] for d in data)
    print('=== ETL Pipeline Summary ===')
    print(f'Total records: {len(data)}')
    print(f'Average salary: ${sum(salaries)/len(salaries):,.2f}')
    print(f'Max salary: ${max(salaries):,}')
    print(f'Min salary: ${min(salaries):,}')
    print(f'Department distribution: {dict(dept_dist)}')

raw_data = extract()
transformed = transform(raw_data)
load(transformed)

print('\n--- ELT Version ---')
# ELT: load raw data first, then transform
def elt_load():
    raw_json = json.dumps(extract())
    print(f'Loaded {len(json.loads(raw_json))} raw records to data lake')
    return json.loads(raw_json)

def elt_transform(data):
    return transform(data)

raw = elt_load()
processed = elt_transform(raw)
load(processed)

## Part 2: API Data Extraction — Solutions

In [ ]:
import time
from collections import deque

# Exercise 2.1: API Client with Pagination
class PaginatedAPIClient:
    def __init__(self, total_records=250, page_size=50):
        self.total_records = total_records
        self.page_size = page_size
        self.total_pages = (total_records + page_size - 1) // page_size
        self.call_timestamps = deque()
    
    def _simulate_page(self, page):
        start = (page - 1) * self.page_size
        end = min(start + self.page_size, self.total_records)
        return [{'id': i, 'value': f'item_{i}'} for i in range(start, end)]
    
    def _rate_limit(self, max_calls=5, window=10):
        now = time.time()
        while self.call_timestamps and self.call_timestamps[0] < now - window:
            self.call_timestamps.popleft()
        if len(self.call_timestamps) >= max_calls:
            sleep_time = self.call_timestamps[0] + window - now
            if sleep_time > 0:
                print(f'  Rate limit: sleeping {sleep_time:.1f}s')
                time.sleep(sleep_time)
        self.call_timestamps.append(time.time())
    
    def fetch_all(self):
        all_data = []
        for page in range(1, self.total_pages + 1):
            for attempt in range(3):
                try:
                    self._rate_limit()
                    data = self._simulate_page(page)
                    all_data.extend(data)
                    print(f'  Fetched page {page}/{self.total_pages} ({len(data)} records)')
                    break
                except Exception as e:
                    print(f'  Retry {attempt + 1} for page {page}: {e}')
                    time.sleep(2 ** attempt)
        return all_data

client = PaginatedAPIClient()
data = client.fetch_all()
print(f'\nTotal records collected: {len(data)}')

In [ ]:
# Exercise 2.2: Authenticated API Client
class AuthenticatedAPIClient:
    def __init__(self):
        self.token = 'initial_token'
        self.session = requests.Session()
        self.session.headers.update({'Authorization': f'Bearer {self.token}'})
    
    def _refresh_token(self):
        print('  Refreshing token...')
        self.token = 'new_token_' + str(int(time.time()))
        self.session.headers.update({'Authorization': f'Bearer {self.token}'})
    
    def get(self, url):
        # Simulate auth handling
        if self.token == 'initial_token':
            print('  Token expired, refreshing...')
            self._refresh_token()
        response = self.session.get(url)
        return {'status': 200, 'data': 'authenticated_data'}

client = AuthenticatedAPIClient()
resp = client.get('https://api.example.com/data')
print(f'Response: {resp}')

## Part 3: Data Formats — Solutions

In [ ]:
import io

# Exercise 3.1: Format Comparison
np.random.seed(42)
df_formats = pd.DataFrame({
    'id': range(5000),
    'numeric1': np.random.randn(5000),
    'numeric2': np.random.randn(5000) * 100,
    'category': np.random.choice(['A', 'B', 'C', 'D'], 5000),
    'datetime': pd.date_range('2024-01-01', periods=5000, freq='h'),
    'text': [f'description_{i}' for i in range(5000)]
})

# CSV
buf_csv = io.BytesIO()
df_formats.to_csv(buf_csv)
size_csv = buf_csv.tell()

# Parquet snappy
buf_snappy = io.BytesIO()
df_formats.to_parquet(buf_snappy, compression='snappy')
size_snappy = buf_snappy.tell()

# Parquet gzip
buf_gzip = io.BytesIO()
df_formats.to_parquet(buf_gzip, compression='gzip')
size_gzip = buf_gzip.tell()

print('=== Format Size Comparison ===')
print(f'CSV:          {size_csv:>8,} bytes')
print(f'Parquet+sna:  {size_snappy:>8,} bytes ({(1 - size_snappy/size_csv)*100:.0f}% smaller)')
print(f'Parquet+gzip: {size_gzip:>8,} bytes ({(1 - size_gzip/size_csv)*100:.0f}% smaller)')

# Verify integrity
buf_snappy.seek(0); df_back = pd.read_parquet(buf_snappy)
assert len(df_back) == 5000
assert df_back['id'].tolist() == df_formats['id'].tolist()
print('\nData integrity verified: all formats read back correctly')

In [ ]:
# Exercise 3.2: Avro Schema Evolution
original_schema = {
    'type': 'record',
    'name': 'User',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'name', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'signup_date', 'type': 'string'}
    ]
}

evolved_schema = {
    'type': 'record',
    'name': 'User',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'name', 'type': 'string'},
        {'name': 'email', 'type': 'string'},
        {'name': 'signup_date', 'type': 'string'},
        {'name': 'age', 'type': ['null', 'int'], 'default': None},
        {'name': 'preferences', 'type': ['null', {'type': 'map', 'values': 'string'}], 'default': None}
    ]
}

# Write with old schema
old_records = [
    {'user_id': 'u1', 'name': 'Alice', 'email': 'alice@example.com', 'signup_date': '2024-01-01'},
    {'user_id': 'u2', 'name': 'Bob', 'email': 'bob@example.com', 'signup_date': '2024-01-15'}
]

buf_avro = io.BytesIO()
fastavro.writer(buf_avro, original_schema, old_records)
buf_avro.seek(0)

reader = fastavro.reader(buf_avro, reader_schema=evolved_schema)
print('=== Avro Schema Evolution ===')
for record in reader:
    print(f'  Read with evolved schema: {record}')

## Part 4: Airflow DAG — Solutions

In [ ]:
import time as tm

# Exercise 4.1: Simulated DAG
class TaskNode:
    def __init__(self, name, func, deps=None):
        self.name = name
        self.func = func
        self.deps = deps or []
        self.executed = False
        self.result = None
        self.duration = 0

def simulate_dag(tasks):
    order = []
    while len(order) < len(tasks):
        for t in tasks:
            if not t.executed and all(d.executed for d in t.deps):
                start = tm.time()
                t.result = t.func()
                t.duration = tm.time() - start
                t.executed = True
                order.append(t.name)
    return order

def fetch(): return {'rows': 1000}
def clean(): return {'rows': 950}
def validate(): return {'status': 'passed'}
def compute(): return {'features': ['f1', 'f2', 'f3']}
def store(): return {'path': 's3://bucket/features/'}

t1 = TaskNode('fetch', fetch)
t2 = TaskNode('clean', clean, [t1])
t3 = TaskNode('validate', validate, [t2])
t4 = TaskNode('compute', compute, [t3])
t5 = TaskNode('store', store, [t4])

exec_order = simulate_dag([t1, t2, t3, t4, t5])
print('===== DAG Execution Order =====')
for i, name in enumerate(exec_order):
    t = next(t for t in [t1, t2, t3, t4, t5] if t.name == name)
    print(f'  {i+1}. {name} ({t.duration:.3f}s) -> {t.result}')

In [ ]:
# Exercise 4.2: XCom Simulation
context = {}

def task_extract(context):
    context['pushed'] = context.get('pushed', {})
    context['pushed']['extract'] = {'feature_names': ['age', 'income', 'score'], 'row_count': 1000}
    return context['pushed']['extract']

def task_analyze(context):
    extracted = context['pushed']['extract']
    stats = {f: {'mean': 42, 'std': 10} for f in extracted['feature_names']}
    context['pushed']['analyze'] = {'stats': stats, 'total_rows': extracted['row_count']}
    return context['pushed']['analyze']

def task_report(context):
    extracted = context['pushed']['extract']
    analyzed = context['pushed']['analyze']
    report = f"""
    === FEATURE PIPELINE REPORT ===
    Rows processed: {extracted['row_count']}
    Features: {', '.join(extracted['feature_names'])}
    Stats: {analyzed['stats']}
    Status: COMPLETE
    """
    return report

# Execute in DAG order
task_extract(context)
task_analyze(context)
report = task_report(context)
print('===== XCom Data Passing =====')
print(report)

## Part 5: Data Quality — Solutions

In [ ]:
import re

# Exercise 5.1: Custom Expectation Suite
class Expectation:
    def __init__(self, name, check_fn):
        self.name = name
        self.check_fn = check_fn
    def validate(self, df):
        passed, msg = self.check_fn(df)
        return {'expectation': self.name, 'passed': passed, 'message': msg}

class ExpectationSuite:
    def __init__(self, name):
        self.name = name
        self.expectations = []
    def add(self, exp):
        self.expectations.append(exp)
    def run(self, df):
        results = [e.validate(df) for e in self.expectations]
        all_passed = all(r['passed'] for r in results)
        return {'suite': self.name, 'success': all_passed, 'results': results}

# Create test data with intentional violations
df_quality = pd.DataFrame({
    'customer_id': ['c1', 'c2', 'c3', 'c4', 'c5', None, 'c1'],
    'age': [25, -5, 130, 35, 45, 30, 25],
    'email': ['a@b.com', 'notanemail', 'c@d.com', 'e@f.com', 'g@h.com', 'i@j.com', 'a@b.com'],
    'signup_date': pd.to_datetime(['2024-01-01', '2024-02-15', '2025-06-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-01-01'])
})

suite = ExpectationSuite('customer_quality')
suite.add(Expectation('id_not_null', lambda df: (df['customer_id'].isna().sum() == 0, f'{df["customer_id"].isna().sum()} null ids')))
suite.add(Expectation('id_unique', lambda df: (df['customer_id'].nunique() == len(df), f'{len(df) - df["customer_id"].nunique()} duplicates')))
suite.add(Expectation('age_range', lambda df: (((df['age'] >= 0) & (df['age'] <= 120)).all(), f'{((df["age"] < 0) | (df["age"] > 120)).sum()} age violations')))
suite.add(Expectation('email_format', lambda df: (df['email'].apply(lambda x: bool(re.match(r'^[\w.+-]+@[\w-]+\.[\w.]+$', str(x)))).all(), 'invalid emails found')))
suite.add(Expectation('future_date', lambda df: ((df['signup_date'] <= pd.Timestamp.now()).all(), 'future dates found')))
suite.add(Expectation('row_count', lambda df: (50 <= len(df) <= 10000, f'rows={len(df)}')))

result = suite.run(df_quality)
print('===== Data Quality Results =====')
print(f'Suite: {result["suite"]}')
print(f'Overall: {"PASS" if result["success"] else "FAIL"}')
for r in result['results']:
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f'  [{status}] {r["expectation"]}: {r["message"]}')

## Part 6: S3 Operations — Solutions

In [ ]:
import boto3
from moto import mock_s3

@mock_s3
def demo_feature_store():
    client = boto3.client('s3', region_name='us-east-1')
    
    # Create bucket with versioning
    s3 = boto3.resource('s3', region_name='us-east-1')
    bucket = s3.create_bucket(Bucket='ml-feature-store')
    bucket.Versioning().enable()
    
    # Upload daily features
    dates = ['2024/01/01', '2024/01/02', '2024/01/03']
    for date in dates:
        key = f'features/{date}/v1.parquet'
        content = f'feature_data_for_{date}'
        client.put_object(Bucket='ml-feature-store', Key=key, Body=content.encode())
        print(f'Uploaded: {key}')
    
    # List available feature dates
    response = client.list_objects_v2(Bucket='ml-feature-store', Prefix='features/')
    print('\nAvailable feature files:')
    for obj in sorted(response.get('Contents', []), key=lambda x: x['Key']):
        print(f'  {obj["Key"]} ({obj["Size"]} bytes)')
    
    # Download latest
    latest_key = 'features/2024/01/03/v1.parquet'
    response = client.get_object(Bucket='ml-feature-store', Key=latest_key)
    print(f'\nDownloaded {latest_key}: {response["Body"].read().decode()}')

demo_feature_store()